# Lax corrections to the linearly polarized Gaussian beam

This notebook is a lean replacement for the development path in `transform_test2.ipynb`.
It keeps the final three-plane visualization and replaces the hand-written Gaussian magnetic field by a field generated from a paraxial vector-potential seed using `lax_series.py`.

The purpose of this stage is diagnostic:

- keep the same linearly polarized Gaussian as the paraxial reference;
- generate its nonparaxial Lax corrections;
- inspect either the cumulative field through order $n$ or the isolated contribution at order $n$;
- retain the existing rotation, offset, slice, time, and Cartesian-component controls.

The Lax construction itself is monochromatic. The old temporal Gaussian gate is retained only as an **external visualization envelope**, applied after the monochromatic field has been constructed.

## 1. Imports and physical parameters

The local visualization coordinates are $(x_0,x_1,x_2)$, with propagation along $x_0$.
The Lax backend uses $(X,Y,Z)$, with propagation along $Z$, so

$$
X=\frac{x_1}{w_0},\qquad
Y=\frac{x_2}{w_0},\qquad
Z=\frac{x_0}{z_R}.
$$

With $\epsilon=2/(kw_0)$ and $z_R=kw_0^2/2$, the physical derivative scales are

$$
\partial_{x_1}=\frac{k\epsilon}{2}\partial_X,\qquad
\partial_{x_2}=\frac{k\epsilon}{2}\partial_Y,\qquad
\partial_{x_0}=\frac{k\epsilon^2}{2}\partial_Z.
$$

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


from lax_series import (
    lax_expand_vector,
    paraxial_residual,
    compile_vector_field,
)

sp.init_printing()

# ------------------------------------------------------------------
# Physical parameters: kept close to transform_test2.ipynb
# ------------------------------------------------------------------
l0 = 2.0 * np.pi
k0 = 1.0
omega = 1.0
waist = 0.65*l0
B0 = 1.0

zR = k0 * waist**2 / 2.0
eps0 = 2.0 / (k0 * waist)

# Plotting / simulation extents
Lsim = [5.0 * l0, 4.0 * l0, 2.5 * l0]
t0 = l0
Tsim = 18.0 * t0

# External temporal gate retained from the old visualizer.
fwhm = 100.0 * t0 # 6.0

def time_envelope(t):
    sigma = (0.5 * fwhm)**2 / np.log(2.0)
    return np.exp(-(t**2) / sigma)

print(f"epsilon = {eps0:.6f}")
print(f"zR      = {zR:.6f}")

## 2. Paraxial seed and Lax-generated vector potential

For the carrier convention $\exp(ikx_0-i\omega t)$, use

$$
\psi_0(X,Y,Z)=\frac{1}{1+iZ}
\exp\!\left[-\frac{X^2+Y^2}{1+iZ}\right].
$$

To reproduce the old dominant $B_{x_1}$ Gaussian at zeroth order, the backend vector potential is chosen along $Y$:

$$
\mathbf A^{(0)}=
\left(0,\; i\frac{B_0}{k}\psi_0,\;0\right).
$$

The carrier part of $\mathbf B$ then gives $B_X^{(0)}=-ikA_Y^{(0)}=B_0\psi_0$. Since backend $X$ corresponds to local $x_1$, this is the old linearly polarized Gaussian in the paraxial limit.

In [ ]:
I = sp.I
X, Y, Z = sp.symbols("X Y Z", real=True)
eps, k, Bamp = sp.symbols("eps k Bamp", positive=True, real=True)

rho2 = X**2 + Y**2
f = 1 / (1 + I * Z)
psi0 = f * sp.exp(-f * rho2)

A0 = (
    sp.Integer(0),
    I * Bamp * psi0 / k,
    sp.Integer(0),
)

seed_residual = sp.simplify(paraxial_residual(psi0, (X, Y, Z)))
print("paraxial residual of psi0:", seed_residual)

# A contains A^(0) + eps^2 A^(2) + eps^4 A^(4).
A_lax = lax_expand_vector(
    A0,
    coords=(X, Y, Z),
    eps=eps,
    order=4,
)

Ay_expanded = sp.expand(A_lax[1])
Ay_coeff = {
    n: Ay_expanded.coeff(eps, n)
    for n in (0, 2, 4)
}

print("generated A orders:", sorted(Ay_coeff))

## 3. Magnetic-field orders and numerical compilation

For this particular seed only $A_Y$ is nonzero. Rather than repeatedly asking SymPy to series-expand the full curl, we collect the magnetic-field orders directly from the same relation used by the generic backend,

$$
\mathbf B=\nabla\times\mathbf A+ik\,\hat{\mathbf Z}\times\mathbf A.
$$

Writing

$$
A_Y=a_0+\epsilon^2 a_2+\epsilon^4 a_4,
$$

the field through $O(\epsilon^5)$ separates cleanly into even transverse and odd longitudinal contributions. The expressions below are the **actual order contributions**, including the corresponding power of $\epsilon$.

The notebook compiles both:

- **cumulative**: $\sum_{j=0}^{n}\epsilon^j\mathbf B^{(j)}$;
- **isolated correction**: $\epsilon^n\mathbf B^{(n)}$.

This keeps slider updates numerical and avoids symbolic work during plotting.

In [ ]:
a0_A = Ay_coeff[0]
a2_A = Ay_coeff[2]
a4_A = Ay_coeff[4]
zero = sp.Integer(0)

# Backend component order is (B_X, B_Y, B_Z).
# The coefficient expressions below do not yet contain eps**n.
B_coeff = {
    0: (-I * k * a0_A,                                  zero, zero),
    1: (zero,                                             zero, k * sp.diff(a0_A, X) / 2),
    2: (-I * k * a2_A - k * sp.diff(a0_A, Z) / 2,        zero, zero),
    3: (zero,                                             zero, k * sp.diff(a2_A, X) / 2),
    4: (-I * k * a4_A - k * sp.diff(a2_A, Z) / 2,        zero, zero),
    5: (zero,                                             zero, k * sp.diff(a4_A, X) / 2),
}

B_order_expr = {
    n: tuple(eps**n * component for component in B_coeff[n])
    for n in range(6)
}

B_cumulative_expr = {}
running = [zero, zero, zero]
for n in range(6):
    running = [running[j] + B_order_expr[n][j] for j in range(3)]
    B_cumulative_expr[n] = tuple(running)

# The physical parameters are fixed by the parameter cell, so substitute
# them before lambdifying. Re-run this cell after changing waist, k0, or B0.
parameter_subs = {
    eps: eps0,
    k: k0,
    Bamp: B0,
}

B_order_fn = {
    n: compile_vector_field(
        tuple(component.subs(parameter_subs) for component in B_order_expr[n]),
        (X, Y, Z),
    )
    for n in range(6)
}

B_cumulative_fn = {
    n: compile_vector_field(
        tuple(component.subs(parameter_subs) for component in B_cumulative_expr[n]),
        (X, Y, Z),
    )
    for n in range(6)
}

print("compiled B orders:", list(range(6)))

## 4. Numerical field adapter and zeroth-order check

`Bfield_meshed` is the only field interface used by the visualization below. It

1. converts local $(x_0,x_1,x_2)$ coordinates to $(X,Y,Z)$;
2. evaluates the selected Lax view;
3. multiplies by the carrier $\exp(ikx_0-i\omega t)$ and takes the real part;
4. maps backend components $(B_X,B_Y,B_Z)$ to local components $(B_{x_0},B_{x_1},B_{x_2})=(B_Z,B_X,B_Y)$;
5. optionally applies the old temporal gate.

The validation at the end of the cell compares the cumulative order-0 field with the old analytic Gaussian expression.

In [ ]:
def _vectorize_compiled_result(raw, Xn, Yn, Zn):
    """Broadcast scalar symbolic zeros to the mesh shape."""
    shape = np.broadcast(np.asarray(Xn), np.asarray(Yn), np.asarray(Zn)).shape
    return np.stack([
        np.broadcast_to(np.asarray(component, dtype=complex), shape)
        for component in raw
    ], axis=0)


def _evaluate_backend_envelope(Xn, Yn, Zn, order, display_mode):
    if display_mode == "cumulative":
        raw = B_cumulative_fn[order](Xn, Yn, Zn)
    elif display_mode == "correction":
        raw = B_order_fn[order](Xn, Yn, Zn)
    else:
        raise ValueError("display_mode must be 'cumulative' or 'correction'.")

    return _vectorize_compiled_result(raw, Xn, Yn, Zn)


def Bfield_meshed(x, t, order=0, display_mode="cumulative", apply_pulse=True):
    """Return the real local magnetic field with shape (3, ...)."""
    x = np.asarray(x)

    Xn = x[1] / waist
    Yn = x[2] / waist
    Zn = x[0] / zR

    B_backend = _evaluate_backend_envelope(Xn, Yn, Zn, order, display_mode)

    # backend (X,Y,Z) -> local (x1,x2,x0)
    B_local_envelope = np.stack([
        B_backend[2],  # local x0 component
        B_backend[0],  # local x1 component
        B_backend[1],  # local x2 component
    ], axis=0)

    carrier = np.exp(1j * (k0 * x[0] - omega * t))
    B_real = np.real(B_local_envelope * carrier)

    if apply_pulse:
        B_real = B_real * time_envelope(t - x[0])

    return B_real


# --- Reference expression from the old notebook, used only for validation ---
def legacy_gaussian_By(x, t):
    x = np.asarray(x)
    r2 = x[1]**2 + x[2]**2
    w = waist * np.sqrt(1.0 + (x[0] / zR)**2)
    invR = x[0] / (x[0]**2 + zR**2)
    gouy = np.arctan(x[0] / zR)

    phase = omega * t - k0 * x[0] - 0.5 * k0 * r2 * invR + gouy
    envelope = (waist / w) * np.exp(-r2 / w**2) * time_envelope(t - x[0])
    return B0 * envelope * np.cos(phase)


# Small numerical check on one plane.
test0 = np.linspace(-2.0 * l0, 2.0 * l0, 41)
test1 = np.linspace(-1.5 * l0, 1.5 * l0, 39)
T0, T1 = np.meshgrid(test0, test1)
test_coords = np.asarray([T0, T1, np.zeros_like(T0)])

new0 = Bfield_meshed(test_coords, 0.0, order=0, display_mode="cumulative")[1]
old0 = legacy_gaussian_By(test_coords, 0.0)

max_reference_error = np.max(np.abs(new0 - old0))
print(f"max |order-0 Lax field - old Gaussian| = {max_reference_error:.3e}")

## 5. Rotation and offset

Only the final vectorized transformation from the old notebook is retained. The field construction is now independent of the transformation layer.

In [ ]:
def RotM(axis, angle):
    """Rotation matrices in the convention used by transform_test2.ipynb."""
    if axis == 'x':
        return np.asarray([
            [1, 0, 0],
            [0, np.cos(angle), -np.sin(angle)],
            [0, np.sin(angle),  np.cos(angle)],
        ])
    if axis == 'y':
        return np.asarray([
            [ np.cos(angle), 0, np.sin(angle)],
            [0, 1, 0],
            [-np.sin(angle), 0, np.cos(angle)],
        ])
    if axis == 'z':
        return np.asarray([
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle),  np.cos(angle), 0],
            [0, 0, 1],
        ])
    raise ValueError("axis must be 'x', 'y', or 'z'.")


def transform_vector_field_vectorised(R, offset, vector, selection=(0, 1, 2)):
    def vector_field_transformed(x_, t_):
        off = np.asarray(offset).reshape((3,) + (1,) * (x_.ndim - 1))

        x_rot = np.einsum('ij,j...->i...', R, x_ - off)
        vec = vector(x_rot, t_)
        vec_rot = np.einsum('ij,j...->i...', R.T, vec)

        return np.asarray([vec_rot[idx] for idx in selection])

    return vector_field_transformed


def transformed_selected_component(
    coords,
    t,
    theta,
    phi,
    offset,
    selected_component,
    order,
    display_mode,
):
    R = RotM('z', theta) @ RotM('y', phi)

    def selected_lax_field(local_x, local_t):
        return Bfield_meshed(
            local_x,
            local_t,
            order=order,
            display_mode=display_mode,
            apply_pulse=True,
        )

    transformed = transform_vector_field_vectorised(
        R,
        np.asarray(offset),
        selected_lax_field,
        selection=(selected_component,),
    )

    return np.squeeze(transformed(coords, t))

## 6. Three-plane visualization

The three final views are the same geometrical slices as in the last cell of `transform_test2.ipynb`:

- $x_0$-$x_1$ at fixed $x_2$;
- $x_0$-$x_2$ at fixed $x_1$;
- $x_1$-$x_2$ at fixed $x_0$.

Two controls are new:

- **B order**: field order $0,1,\ldots,5$;
- **view**:
  - `cumulative` shows all terms through that order;
  - `correction` shows only the actual $\epsilon^n\mathbf B^{(n)}$ contribution at that order.

Each panel is normalized by $B_0$. Its title reports the plotted maximum, so autoscaling cannot hide the absolute size of a small correction.

In [ ]:
# N=250 reproduces the resolution of the old notebook.
# Reduce it if high-order slider updates feel slow on a local machine.
N = 250
cmap = "RdBu_r"

x0_vals = np.linspace(-Lsim[0], Lsim[0], N)
x1_vals = np.linspace(-Lsim[1], Lsim[1], N)
x2_vals = np.linspace(-Lsim[2], Lsim[2], N)

X01_0, X01_1 = np.meshgrid(x0_vals, x1_vals)
X02_0, X02_2 = np.meshgrid(x0_vals, x2_vals)
X12_1, X12_2 = np.meshgrid(x1_vals, x2_vals)


def _safe_limit(field):
    limit = float(np.max(np.abs(field)))
    return limit if limit > 0.0 else 1.0


def plot_field_3planes(
    t,
    x0,
    x1,
    x2,
    theta,
    phi,
    off0,
    off1,
    off2,
    selected_component,
    order,
    display_mode,
):
    offset = (off0, off1, off2)

    coords01 = np.asarray([
        X01_0,
        X01_1,
        np.full_like(X01_0, x2),
    ])
    coords02 = np.asarray([
        X02_0,
        np.full_like(X02_0, x1),
        X02_2,
    ])
    coords12 = np.asarray([
        np.full_like(X12_1, x0),
        X12_1,
        X12_2,
    ])

    field01 = transformed_selected_component(
        coords01, t, theta, phi, offset,
        selected_component, order, display_mode,
    ) / B0
    field02 = transformed_selected_component(
        coords02, t, theta, phi, offset,
        selected_component, order, display_mode,
    ) / B0
    field12 = transformed_selected_component(
        coords12, t, theta, phi, offset,
        selected_component, order, display_mode,
    ) / B0

    limits = [_safe_limit(field01), _safe_limit(field02), _safe_limit(field12)]
    maxima = [
        float(np.max(np.abs(field01))),
        float(np.max(np.abs(field02))),
        float(np.max(np.abs(field12))),
    ]

    fig, axs = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    component_names = {0: 'x0', 1: 'x1', 2: 'x2'}
    component = component_names[selected_component]
    mode_label = "through" if display_mode == "cumulative" else "isolated"

    panels = [
        (
            axs[0], field01,
            [-Lsim[0], Lsim[0], -Lsim[1], Lsim[1]],
            'x0', 'x1', f'x0-x1, x2={x2:.3f}',
            limits[0], maxima[0],
        ),
        (
            axs[1], field02,
            [-Lsim[0], Lsim[0], -Lsim[2], Lsim[2]],
            'x0', 'x2', f'x0-x2, x1={x1:.3f}',
            limits[1], maxima[1],
        ),
        (
            axs[2], field12,
            [-Lsim[1], Lsim[1], -Lsim[2], Lsim[2]],
            'x1', 'x2', f'x1-x2, x0={x0:.3f}',
            limits[2], maxima[2],
        ),
    ]

    for ax, field, extent, xlabel, ylabel, plane_title, limit, maximum in panels:
        im = ax.imshow(
            field,
            extent=extent,
            origin='lower',
            aspect='auto',
            vmin=-limit,
            vmax=limit,
            cmap=cmap,
        )
        ax.set_title(
            f'{plane_title}\n'
            f'B_{component}, {mode_label} order {order}; '
            f'max |B/B0|={maximum:.3e}'
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

        cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.12)
        cbar.set_label('B / B0')

    plt.show()

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

interact(
    plot_field_3planes,

    t=FloatSlider(
        min=-Tsim,
        max=Tsim,
        step=Tsim / 200,
        value=0.0,
        description='t',
    ),
    x0=FloatSlider(
        min=-Lsim[0],
        max=Lsim[0],
        step=(2 * Lsim[0]) / 200,
        value=0.0,
        description='x0',
    ),
    x1=FloatSlider(
        min=-Lsim[1],
        max=Lsim[1],
        step=(2 * Lsim[1]) / 200,
        value=0.0,
        description='x1',
    ),
    x2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2 * Lsim[2]) / 200,
        value=0.0,
        description='x2',
    ),
    theta=FloatSlider(
        min=-0.5 * np.pi,
        max=0.5 * np.pi,
        step=np.pi / 200,
        value=0.0,
        description='theta',
    ),
    phi=FloatSlider(
        min=-0.5 * np.pi,
        max=0.5 * np.pi,
        step=np.pi / 200,
        value=0.0,
        description='phi',
    ),
    off0=FloatSlider(
        min=-Lsim[0],
        max=Lsim[0],
        step=(2 * Lsim[0]) / 200,
        value=0.0,
        description='off0',
    ),
    off1=FloatSlider(
        min=-Lsim[1],
        max=Lsim[1],
        step=(2 * Lsim[1]) / 200,
        value=0.0,
        description='off1',
    ),
    off2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2 * Lsim[2]) / 200,
        value=0.0,
        description='off2',
    ),
    selected_component=IntSlider(
        min=0,
        max=2,
        step=1,
        value=1,
        description='component',
    ),
    order=IntSlider(
        min=0,
        max=5,
        step=1,
        value=0,
        description='B order',
    ),
    display_mode=Dropdown(
        options=[
            ('cumulative', 'cumulative'),
            ('isolated correction', 'correction'),
        ],
        value='cumulative',
        description='view',
    ),
);